In [45]:
from openai import OpenAI
# Basta puntare al server vLLM locale invece che alle API pubbliche
client = OpenAI(
    base_url="http://localhost:8000/v1",  # <--- UNICA MODIFICA
    api_key="EMPTY",
)

response = client.chat.completions.create(
    model="unsloth/SmolLM3-3B-bnb-4bit",
    messages=[{"role": "user", "content": "Ciao sono Andrea!"}],
    extra_body={"chat_template_kwargs":{"enable_thinking": False}},
)

# Stampa la risposta e i token usati
print(response.choices[0].message.content)
print("\n--- Dettagli Tecnici ---")
print(f"Token totali: {response.usage.total_tokens}")
print(f"Modello usato: {response.model}")

Ciao Andrea! Piacere di conoscerti. Come posso aiutarti oggi?

--- Dettagli Tecnici ---
Token totali: 95
Modello usato: unsloth/SmolLM3-3B-bnb-4bit


In [48]:
#NB NB NB Se devo fare interrogazioni successive devo inserire nella lista di messaggi la risposta dell'assistente e la successiva domanda dell'utente
response = client.chat.completions.create(
    model="unsloth/SmolLM3-3B-bnb-4bit",
    messages=[{"role": "user", "content": "Ciao sono Andrea!"}, {"role":"assistant", "content": response.choices[0].message.content}, {"role": "user", "content": "Qual'è il mio nome?"}],
    extra_body={"chat_template_kwargs":{"enable_thinking": False}},
)
print(response.choices[0].message.content)
print("\n--- Dettagli Tecnici ---")
print(f"Token totali: {response.usage.total_tokens}")
print(f"Modello usato: {response.model}")

Il tuo nome è Andrea! Ho risposto la tua domanda prima. Il tuo nome è Andrea, come ho già detto.

--- Dettagli Tecnici ---
Token totali: 139
Modello usato: unsloth/SmolLM3-3B-bnb-4bit


In [49]:
#Se non aggiungo il risultato precedente alla lista di messaggi il sistema non sa nulla, non mantiene in nessun modo la storia dei messaggi
response = client.chat.completions.create(
    model="unsloth/SmolLM3-3B-bnb-4bit",
    messages=[{"role": "user", "content": "Qual'è il mio nome'"}],
    extra_body={"chat_template_kwargs":{"enable_thinking": True}},
)

# Stampa la risposta e i token usati
print(response.choices[0].message.content)
print("\n--- Dettagli Tecnici ---")
print(f"Token totali: {response.usage.total_tokens}")
print(f"Modello usato: {response.model}")

<think>
Okay, the user wrote "Qual'è il mio nome?" which means "What is my name?" in Italian. I need to respond appropriately. Since I don't have access to personal data or context about the user, I can't provide their actual name. I should explain that I can't access personal information and offer to help with something else related to Italian language or general questions. Let me check if there's a standard way to handle such a request. Maybe I can say something like, "I don't have access to your personal information. If you need help with Italian language, I can assist with that." Or maybe ask if they want to practice a sentence in Italian. I should keep it friendly and helpful.
</think>

I don't have access to your personal information. However, I can help with Italian language practice, grammar, or general questions! For example, could you say a sentence in Italian or translate something for me?

--- Dettagli Tecnici ---
Token totali: 448
Modello usato: unsloth/SmolLM3-3B-bnb-4bit

# Modalità streaming

In [20]:
for chunk in client.chat.completions.create(
    model="unsloth/SmolLM3-3B-bnb-4bit",
    messages=[{"role": "user", "content": "Ciao vLLM!"}],
    extra_body={"chat_template_kwargs":{"enable_thinking": False}},
    stream=True
):
    print(chunk.choices[0].delta.content)
# Stampa la risposta e i token usati



C
iao
!
 Come
 pos
so
 ai
ut
arti
 oggi
?
 Hai
 qualche
 dom
anda
 o
 hai
 bis
og
no
 di
 inform
azioni
 su
 un
 determin
ato
 arg
oment
o
?



## Metriche

2. Le Metriche Chiave (Cosa monitorare)
Le metriche più importanti che vLLM ti restituisce sono:

vllm:gpu_cache_usage_perc: Questa è la metrica "regina". Ti dice quanto è piena la memoria della GPU gestita da PagedAttention.

Perché è utile: Se è al 95%, stai usando la GPU al massimo (ottimo). Se è bassa, puoi aumentare il batch size.

vllm:num_requests_running: Quante richieste sta elaborando in parallelo in questo istante.

vllm:num_requests_waiting: Quante richieste sono in coda (perché la GPU è piena).

Perché è utile: Se questo numero sale, ti serve un'altra GPU.

vllm:time_to_first_token_seconds: Quanto tempo passa da quando l'utente preme invio a quando vede la prima parola (latenza percepita).

vllm:generation_tokens_total: La velocità pura (throughput).

In [29]:
import requests
requests.get('http://localhost:8000/metrics').text

'# HELP python_gc_objects_collected_total Objects collected during gc\n# TYPE python_gc_objects_collected_total counter\npython_gc_objects_collected_total{generation="0"} 23335.0\npython_gc_objects_collected_total{generation="1"} 3837.0\npython_gc_objects_collected_total{generation="2"} 1389.0\n# HELP python_gc_objects_uncollectable_total Uncollectable objects found during GC\n# TYPE python_gc_objects_uncollectable_total counter\npython_gc_objects_uncollectable_total{generation="0"} 0.0\npython_gc_objects_uncollectable_total{generation="1"} 0.0\npython_gc_objects_uncollectable_total{generation="2"} 0.0\n# HELP python_gc_collections_total Number of times this generation was collected\n# TYPE python_gc_collections_total counter\npython_gc_collections_total{generation="0"} 1608.0\npython_gc_collections_total{generation="1"} 146.0\npython_gc_collections_total{generation="2"} 10.0\n# HELP python_info Python platform information\n# TYPE python_info gauge\npython_info{implementation="CPython"